# Company Specific Risk Premium Calculation (CSRP)

**Objective**
The valuation process for companies, in the event of M&As. 
Adapting the automated reviewer pipeline to compile CRSP risk factors for a sample company, and produce associated documentation.

Since we lack financial statements data for private companies, we first build a PoC with publicly traded companies and publicly available financial statements, to demonstrate how the pipeline could work in practice.


In [3]:
import os
import json
import src
from pathlib import Path
#from src.llm import *
from src.perform_ingestion import *

#here = os.path.dirname(os.path.dirname(os.path.realpath(__file__)))
#sample = os.path.join(here, "data", "SAAF.pdf")
sample = Path(os.getcwd()) / "data" / "SAAF.pdf"

doc = ingest(sample)
print(f"file:        {os.path.basename(doc.path)}")
print(f"pages:       {doc.page_count}")
print(f"toc_source:  {doc.toc_source}")
print(f"sections:    {len(doc.sections)}")
print()
print("First 12 TOC entries (indent = level):")
for s in doc.sections[:12]:
    indent = "  " * (s["level"] - 1)
    print(f"  {indent}[p{s['page_start']}-{s['page_end']}] {s['title']}")
print()

target = "３事業等のリスク"
text = doc.section_text(target)
print(f"section_text('{target}') -> {len(text)} chars, first 200:")
print(text[:200])

file:        SAAF.pdf
pages:       111
toc_source:  embedded
sections:    86

First 12 TOC entries (indent = level):
  [p1-1] 表紙
  [p2-2] 本文
  [p2-103] 第一部企業情報
    [p2-10] 第１企業の概況
      [p2-3] １主要な経営指標等の推移
      [p4-4] ２沿革
      [p5-6] ３事業の内容
      [p7-8] ４関係会社の状況
      [p9-10] ５従業員の状況
    [p11-26] 第２事業の状況
      [p11-14] １経営方針、経営環境及び対処すべき課題等
      [p15-16] ２サステナビリティに関する考え方及び取組

section_text('３事業等のリスク') -> 4179 chars, first 200:
３【事業等のリスク】
　有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が連結会社の財政状態、経営成
績およびキャッシュ・フローの状況に重要な影響を与える可能性があると認識している主要なリスクは、以下のとお
りであります。
　当社グループでは、これらのリスク発生の可能性を認識したうえで、発生の回避および発生時の対応に全力で対処
する方針でありますが、当社株式に対する投資判


# Perform Review

In [ ]:
from src.perform_review import *
Record = dict[str, Any]

#here = os.path.dirname(os.path.dirname(os.path.realpath(__file__)))
#taxonomy = json.load(open(os.path.join(here, "taxonomy.json")))["csrp_framework"]
here = Path(os.getcwd())
taxonomy = json.load(open(Path(os.getcwd()) / "taxonomy.json"))["csrp_framework"]

# Synthetic structured records exercise the deterministic path with no API.
def struct(flag: bool | None, value: Any) -> Record:
    return {"value": value, "flag": flag, "evidence_en": "synthetic",
            "citation": {"citation_type": "structured"}, "layer": "structured",
            "status": "scored"}

records: dict[str, Record] = {
    "FIN_01": struct(True, 6.9),    # breach -> 4
    "FIN_02": struct(False, 6.73),  # clear  -> 2
    "FIN_03": struct(True, "57% ≤24m"),
}
scored = score_signals(taxonomy, records, client=None)
print("threshold scoring:")
for sid in ("FIN_01", "FIN_02", "FIN_03"):
    r = scored[sid]
    print(f"  {sid}: score={r['score']} scored_by={r['scored_by']}")

agg = aggregate(taxonomy, scored)
scored_cats = {k: v for k, v in agg["category_scores"].items() if v is not None}
print(f"\ncategory_scores (scored only): {scored_cats}")
print(f"composite={agg['composite_score']}  -> "
        f"{agg['csrp_bps_low']}-{agg['csrp_bps_high']} bps")
print(f"unscored signals: {len(agg['unscored_signals'])}")

print("\ncomposite_to_bps checks:")
for c in (1.2, 2.4, 3.2, 4.5):
    print(f"  {c} -> {composite_to_bps(taxonomy, c)} bps")

# Live end-to-end scoring (costs tokens): python -m src.perform_review --llm
#if "--llm" in sys.argv:
from src.perform_ingestion import ingest
from src.perform_extraction import Extractor, StructuredExtractor
from src.llm import create_client

client, model = create_client(DEFAULT_MODEL)
doc = ingest(os.path.join(here, "data", "SAAF.pdf"))
ext = Extractor(doc, taxonomy, client=client, model=model, company_name="SAAF Co., Ltd.")
by_id = {c["id"]: c for c in taxonomy["categories"]}
nlp = ext.extract_category(by_id["MGMT"])  # one NLP category to keep it cheap
structured = StructuredExtractor(doc, client=client, model=model, company_name="SAAF Co., Ltd.").run()
merged = merge_records(nlp, structured)
result = score(taxonomy, merged, client=client, model=model, company_name="SAAF Co., Ltd.")
print("\n### live scoring (MGMT + structured) ###")
for sid, r in result["signals"].items():
    if r["score"] is not None:
        print(f"  {sid}: {r['score']} ({r['scored_by']})")
print(f"composite={result['composite_score']} -> {result['csrp_bps_low']}-{result['csrp_bps_high']} bps")

threshold scoring:
  FIN_01: score=4 scored_by=threshold
  FIN_02: score=2 scored_by=threshold
  FIN_03: score=4 scored_by=threshold

category_scores (scored only): {'FIN': 3.33}
composite=3.33  -> 200-300 bps
unscored signals: 54

composite_to_bps checks:
  1.2 -> (0, 50) bps
  2.4 -> (100, 150) bps
  3.2 -> (200, 300) bps
  4.5 -> (400, 500) bps
Using Anthropic API with model claude-sonnet-4-6.
